In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import glob

import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_debug_nans", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.plotting_utils import plot_sequence
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.utils.density_estimation import build_grid
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.sets.reachable_set import approximate_reachable_set
from dmpe.utils.sets.control_invariant_set import approximate_control_invariant_set

from dmpe.utils.sets.reachable_set import save_results as save_results_rs
from dmpe.utils.sets.control_invariant_set import save_results as save_results_ci
from dmpe.utils.sets.shared import load_results, DiscretizedSet, SlicedSet, load_discretized_set, save_discretized_set

In [ ]:
from enum import Enum
class Systems(Enum):
    FLUID_TANK = 1
    PENDULUM = 2
    CART_POLE = 3

## load $S^{(x,u)}$

In [ ]:
sys_name = Systems.FLUID_TANK

In [ ]:
if sys_name == Systems.FLUID_TANK:
    env, penalty_function, featurize, _ = setup_fluid_tank_env()
    S_xu = load_discretized_set(DataPaths().reach_ci_experiments / "fluid_tank_S_xu_666a769d-8c1b-4b.json")
elif sys_name == Systems.PENDULUM:
    env, penalty_function, featurize, _ = setup_pendulum_env()
    S_xu = load_discretized_set(DataPaths().reach_ci_experiments / "pendulum_S_xu_02430b86-ae0d-42.json")
elif sys_name == Systems.CART_POLE:
    env, penalty_function, featurize, _ = setup_cart_pole_env()
    S_xu = load_discretized_set(DataPaths().reach_ci_experiments / "cart_pole_S_xu_8744b5d5-30e6-4b.json")

In [ ]:
fig, axs = S_xu.visualize(use_contourf=False)

In [ ]:
S_xu.grid

## load experiment:

In [ ]:
from dmpe.evaluation.exp_data_model_learning import ModelExpDataResult

In [ ]:
data_path = DataPaths().model_learning_cs_out / str(sys_name.name).lower() / "2step"
result_path = glob.glob(str(data_path) + "/*.eqx")[2]

result = ModelExpDataResult.from_file(
    filename=result_path,
    model_class=NeuralEulerODECartpole,
)
data_points = jnp.concatenate([result.observations, result.actions], axis=-1)

In [ ]:
fig, axs = S_xu.visualize(use_contourf=True)

if sys_name == Systems.FLUID_TANK:
    axs.scatter(result.observations, result.actions, s=4)
    # axs.scatter(S_xu.grid[:, 0], S_xu.grid[:, 1], s=4, c="r")
else:
    n_features = env.reset(env.env_properties)[0].shape[-1] + env.action_dim
    
    for i in range(n_features):
        for j in range(n_features):
            axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.1, c="r")

## check fill:

In [ ]:
# grid = S_xu.grid
if sys_name == Systems.CART_POLE:
    points_per_dim = 7  
elif sys_name == Systems.PENDULUM:
    points_per_dim = 15
else:
    points_per_dim = 25
dim = S_xu.grid.shape[-1]
grid = build_grid(dim, -1, 1, points_per_dim)

grid_spacing = jnp.abs(grid[0] - grid[1])[-1] / 2

In [ ]:
def check_point_to_point(grid_point, point, grid_spacing):
    lower = grid_point - grid_spacing
    upper = grid_point + grid_spacing

    return jnp.logical_and(jnp.all(lower < point), jnp.all(upper > point))

def check_point_to_grid(grid, point, grid_spacing):
    return eqx.filter_vmap(check_point_to_point, in_axes=(0, None, None))(grid, point, grid_spacing)

def check_points_to_grid(grid, points, grid_spacing):
    return eqx.filter_vmap(check_point_to_grid, in_axes=(None, 0, None))(grid, points, grid_spacing)

In [ ]:
if sys_name == Systems.CART_POLE:
    chunk_size = 100
    out = []
    n = data_points.shape[0]
    for i in tqdm(jnp.arange(0, n, chunk_size)):
        out.append(jnp.any(check_points_to_grid(grid, data_points[i : min(i + chunk_size, n)], grid_spacing), axis=0))
    out = jnp.any(jnp.vstack(out), axis=0)
else:
    out = jnp.any(check_points_to_grid(grid, data_points, grid_spacing), axis=0)

In [ ]:
filled_set = DiscretizedSet(grid=grid, mask=out, unflattened_shape=tuple([points_per_dim] * dim))
fig, axs = filled_set.visualize()
plt.show()

fig, axs = filled_set.visualize()
if sys_name == Systems.FLUID_TANK:
    axs.scatter(result.observations, result.actions, s=4)
    # axs.scatter(grid[:, 0], grid[:, 1], s=4, c="r")
else:
    n_features = env.reset(env.env_properties)[0].shape[-1] + env.action_dim
    
    for i in range(n_features):
        for j in range(n_features):
            axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")

## compute percentage:

In [ ]:
from dmpe.utils.sets.shared import check_in_set

In [ ]:
chunk_size = 1000

out = []

n = grid.shape[0]
for i in tqdm(jnp.arange(0, n, chunk_size)):
    out.append(eqx.filter_vmap(check_in_set, in_axes=(0, None, None))(grid[i : min(i + chunk_size, n)], S_xu.mask, S_xu.grid))
subsampled_set = DiscretizedSet(grid=grid, mask=jnp.hstack(out), unflattened_shape=tuple([points_per_dim] * dim))
# subsampled_set.visualize()

In [ ]:
print(f"Reachable and safe: {jnp.sum(subsampled_set.mask) / subsampled_set.mask.shape[0] * 100} %.")
print(f"Reached of all: {jnp.sum(filled_set.mask) / filled_set.mask.shape[0] * 100} %.")
print(f"Reached of safe: {jnp.sum(filled_set.mask) / jnp.sum(subsampled_set.mask) * 100} %.")

In [ ]:
from dmpe.utils.metrics import diced_fill

In [ ]:
diced_fill(
    data_points=jnp.concatenate([result.observations, result.actions], axis=-1),
    support_points=grid,
    support_spacing=grid_spacing,
)

In [ ]:
subsampled_set

In [ ]:
diced_fill(
    data_points=jnp.concatenate([result.observations, result.actions], axis=-1),
    support_points=subsampled_set.grid[subsampled_set.mask],
    support_spacing=grid_spacing,
)

---

In [ ]:
from dmpe.evaluation.metrics_utils import default_df

In [ ]:
default_df(result.observations, result.actions, points_per_dim=points_per_dim)

In [ ]:
data_path = DataPaths().model_learning_cs_out / str(sys_name.name).lower() / "2step"
result_paths = glob.glob(str(data_path) + "/*.eqx")

df_values = []
model_errors = []

for result_path in tqdm(result_paths):
    result = ModelExpDataResult.from_file(
        filename=result_path,
        model_class=NeuralEulerODECartpole,
    )
    df_values.append(default_df(result.observations, result.actions, points_per_dim=points_per_dim))
    
    model_errors.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8.89, 4.5))

ax.scatter(
    1 - jnp.stack(df_values), model_errors, s=25, marker="x"
)
ax.set_yscale('log')